### Sensitivy Region

#### 20% Foreground

In [ ]:
#### Optimistic Foreground ####



### User Variables ###

constraint_value = 0.05
constraint_label = "20%"
# constraints = np.array([0.00000025])
N_antenna = 10
dnu=1e6
dt=100*3600
N_extractions = 100
N_TS = 10000   # Number of curves in the training set
amplitudes = np.arange(0,20,0.5)

evidence_cutoff = 0

### Code Backend ###
plot=False
verbose = False
n_LSTs = 10 # this should match the number of foreground_training_sets you've made
## Spectral Index Map Variable
n_regions = 8   # number of regions that are created for your multi-region model
temp1 = 20     # lower frequency that is used for the temperature for the spectral index of the spectral index map
temp2 = 50     # upper frequency
f_low = temp1   # Only concerns the residual maps that are displayed through spectral_index_map_visualizations (not important for this)
f_high = temp2  # Only concerns the residual maps that are displayed through spectral_index_map_visualizations (not important for this)
NSIDE = 32
## Monochromatic Beam Variables.
N_beam = 1   # Number of beams to create. I'm not varying the beam right now, but this may be helpful later.
low_std = 0.5
high_std = low_std
## Synchrotron Foreground Variables
ylim = (-0.07,0.03)

quantity = AttributeQuantity("log_evidence")



constraint=str(constraint_value)[2:]

parameter_variation = [0,constraint_value,0]
sky_maps = master_foreground_array
sky_maps_truncated = sky_maps[:,4:49,:]  # the trucated sky maps that map the frequency range of frequencies (though it may not match the number of bins)
frequency_range_truncated = np.arange(5,50)  # Truncated frequency range that matches the sky map frequencies
# Pylinex Extraction Variables
manual_systematics_terms = 0
manual_signal_terms = 0
num_sig_vectors = 3 # maximum number of signal modes to use in the extractions
num_sys_vectors = 75    # maximum number of systematic modes to use in the extraction
num_basis_vectors = 75  # number of basis vectors to create for the root bases (make sure its as big as the largest num_sys or num_sig_vectors number)
## signal that is used as the input to the observation data: ##
parameter_array = np.array([[omR0,0],[omM0h2,0],[omK0,0],[omL0,0],[omB0h2,0/h**2],[H0,0]])

##  ##
signal_training_set = lambdaCDM_training_set[:,16::4]  ### indexing is needed to get it into the correct shape for the extraction. Training set used in the pylinex fit
IC = "DIC"

master_chi_squared_array = np.zeros((len(amplitudes),N_extractions))
master_psi_squared_array = np.zeros((len(amplitudes),N_extractions))
master_term_number_array =np.zeros((len(amplitudes),N_extractions))
master_evidence_array = np.zeros((len(amplitudes),N_extractions))
master_term_number_array_signal =np.zeros((len(amplitudes),N_extractions))

# array for the Lambda CDM extraction (not the null signal, that's the ones above) 
master_chi_squared_array_CDM = np.zeros((len(amplitudes),N_extractions))
master_psi_squared_array_CDM = np.zeros((len(amplitudes),N_extractions))
master_term_number_array_CDM = np.zeros((len(amplitudes),N_extractions))
master_evidence_array_CDM = np.zeros((len(amplitudes),N_extractions))
master_term_number_array_CDM_signal = np.zeros((len(amplitudes),N_extractions))


#Step 1: Create the spectral index map that will govern the B_values of the beam (since they govern the indices of the regions).
# NPIX = hp.nside2npix(NSIDE)
# spectral_maps = np.zeros((n_LSTs,NPIX))
# for l in range(n_LSTs):
#     spectral_map = py21cmsig.spectral_index_map(f_low,f_high,n_regions,temp1,temp2,NSIDE,ULSA_direction_32 = master_foreground_array[l])
#     spectral_maps[l] = spectral_map

# f = h5py.File("/home/dbarker7752/21_cm_group/Exotic_Models_Data/spectral_index_map_10to50MHz_32bit","w")
# f.create_dataset("spectral maps",data=spectral_maps)
# f.close()

if NSIDE != 32:
    raise ValueError ("The saved spectral maps are only 32 bit right now.")
hdf5 = h5py.File("/home/dbarker7752/21_cm_group/Exotic_Models_Data/spectral_index_map_10to50MHz_32bit","r")
spectral_maps = np.array(hdf5["spectral maps"])   # I'v visually varified that these make sense

BTS_curves_monochromaticbeam = np.zeros((n_LSTs,len(frequencies)))
# # Step 2: Create the monochromatic beam
for l in np.arange(n_LSTs):
    BTS_curves_monochromaticbeam_raw = (py21cmsig.gaussian_beams(frequency_range_truncated,low_std,resolution=NSIDE,monochromatic_mode=True)[0]*sky_maps_truncated).sum(axis=2)
    BTS_interpolator = scipy.interpolate.CubicSpline(frequency_range_truncated,BTS_curves_monochromaticbeam_raw[l])
    BTS_curves_monochromaticbeam[l] = BTS_interpolator(frequencies)

# creates the B values
N=1
monochromatic_gaussian_B_values = np.zeros((n_LSTs,N,len(frequencies),n_regions))
gaussian_beam_element = py21cmsig.gaussian_beams(frequencies,0.5,NSIDE,monochromatic_mode=True)

for l in range(n_LSTs):
    patch=perses.models.PatchyForegroundModel(frequencies,spectral_maps[l],n_regions) # define the regional patches
    region_indices = patch.foreground_pixel_indices_by_region_dictionary
    for r in range(n_regions):
        monochromatic_gaussian_B_values[l,0,:,r] = gaussian_beam_element[:,region_indices[r]].sum(axis=1)
# monochromatic_gaussian_B_values = py21cmsig.create_monochromatic_gaussian_beam_training_set(frequencies,N_beam,[low_std,high_std],n_LSTs,n_regions,NSIDE,spectral_maps,plot_extremes = True)[0]
monochromatic_gaussian_parameters = np.array([low_std])
# if len(frequencies) != monochromatic_gaussian_B_values[0][0].shape[0]:
#     raise ValueError ("Your loaded B_values do not have the same length as your frequencies. This will cause problems later")

# # Step 3: Create the beam-weighted foreground training set:
# # Step 3a: Create the BTS_curves to input to the foreground function:

# simulated_foreground_array sets the "true sky" observations that will be compared to the training sets for the extraction. It's your data, basically. 
parameter_variation_true = [0,0,0]   
simulated_foreground_array_10LSTs = np.zeros((n_LSTs,len(frequencies)))
for l in range(n_LSTs):
    simulated_foreground_array_10LSTs[l,:] = py21cmsig.synchrotron_foreground_updated_given_Bvalues(1,n_regions,frequencies,spectral_maps[l],sky_maps[l],parameter_variation_true,\
                                                                                    np.array([monochromatic_gaussian_B_values[l,0]]),np.array([low_std]),\
                                                                                    np.array([BTS_curves_monochromaticbeam[l]]))[0]

# The systematics training set itself
monochro_beam_TS_10LSTs = np.zeros((N_TS,n_LSTs,len(frequencies)))
for l in range(n_LSTs):
    monochro_beam_TS_10LSTs[:,l,:] = py21cmsig.synchrotron_foreground_updated_given_Bvalues(N_TS,n_regions,frequencies,spectral_maps[l],sky_maps[l],parameter_variation,\
                                                                                    np.array([monochromatic_gaussian_B_values[l,0]]),np.array([monochromatic_gaussian_parameters[0]]),\
                                                                                    np.array([BTS_curves_monochromaticbeam[l]]))[0]



    #Pylinex Extractions



    extraction_dict = {}

    ### Pylinex part (doing it manually since we are only using the foreground for this) ###
# for amplitude_index in tqdm(np.arange(len(amplitudes))):
for amplitude_index in np.arange(len(amplitudes)):
    amplitude = amplitudes[amplitude_index]
    print(f"Beginning processing of {amplitude}x amplitude multiplier.")
    for index in tqdm(np.arange(N_extractions)):  
    
        input_signal = fiducial_signal*amplitude
        foreground = monochro_beam_TS_10LSTs[np.random.randint(N_TS)]
        static_monchrome_beam_simulation=py21cmsig.multi_spectra_simulation_run(frequencies,foreground,input_signal,N_antenna,dnu,dt)
        data=static_monchrome_beam_simulation[0]
        noise = static_monchrome_beam_simulation[5]
        signal = static_monchrome_beam_simulation[1]
        systematics = static_monchrome_beam_simulation[2]
        systematics_training_set = monochro_beam_TS_10LSTs
        flattened_systematics_training_set = np.reshape(systematics_training_set, (len(systematics_training_set), -1))
        temperatures_flat = data.flatten()
        noise_level_flat = noise.flatten()
        Ns = len(noise)      # grabs the number of correlated spectra you are using
        names = ["systematics"]        # names of your bases
        bases = [
        TrainedBasis(
            training_set=flattened_systematics_training_set,
            num_basis_vectors=num_basis_vectors,
            error=noise_level_flat,
            expander=NullExpander(),
            mean_translation=True
        )]
        basis_sum = BasisSum(names,bases)
        dimension = [{'systematics' : np.arange(1,num_sys_vectors)}]
        priors = {"systematics_prior" : bases[0].gaussian_prior} 
        meta_fitter = MetaFitter(basis_sum, temperatures_flat, noise_level_flat, quantity, quantity.name, *dimension,**priors,verbose=False)      # creates the meta_fitter object, which is an essential Pylinex object
        logZ_grid = meta_fitter["log_evidence"]

        best_indices = np.unravel_index(
            np.argmax(logZ_grid),
            logZ_grid.shape
        )

        fitter = meta_fitter.fitter_from_indices(best_indices)

        # print("Best term numbers:", fitter.sizes)
        # print("Maximum log evidence:", fitter.log_evidence)
                                                                    
        num_systematics_terms = fitter.sizes["systematics"]                                                
        # plot for the signal extraction
        pylinex_error = fitter.subbasis_channel_error("systematics")
        radiometer_error = noise[0]
        # plt.plot((fitter.subbasis_channel_mean("systematics")-temperatures_flat)/noise_level_flat,label = "residual fit mean",color="blue")
        # print(f"systematic terms used: {num_systematics_terms}")
        # print(f"reduced chi squared: {fitter.reduced_chi_squared}")
        # print(f"reduced psi squared: {fitter.psi_squared}")
        chi_squared = fitter.reduced_chi_squared
        psi_squared = fitter.psi_squared
        term_number = num_systematics_terms

        master_chi_squared_array[amplitude_index][index] = chi_squared
        master_psi_squared_array[amplitude_index][index] = psi_squared
        master_term_number_array[amplitude_index][index]= term_number   
        master_evidence_array[amplitude_index][index]= fitter.log_evidence

            ##### Lambda CDM extractions
        data=static_monchrome_beam_simulation[0]
        noise = static_monchrome_beam_simulation[5]
        signal = static_monchrome_beam_simulation[1]
        systematics = static_monchrome_beam_simulation[2]
        test_extraction_static_monochrome_LCDM = pylinex_extraction(systematics_training_set,signal_training_set,data,noise,signal,frequencies,IC="log_evidence",verbose=verbose,plot=plot,plot_residual=False,num_basis_vectors=num_basis_vectors,num_sys_vectors = num_sys_vectors, \
                        num_sig_vectors = num_sig_vectors, title = "Pylinex Extraction", ignore_IC = False,ylim=None,man_sys_terms=manual_systematics_terms,man_sig_terms=manual_signal_terms,man_sys_terms_lower=0,man_sig_terms_lower=0,multi_spectra=True,priors=True,covariance_expansion_factor=1\
                            ,evidence_cutoff=evidence_cutoff)



        master_chi_squared_array_CDM[amplitude_index][index] = test_extraction_static_monochrome_LCDM[0].reduced_chi_squared
        master_psi_squared_array_CDM[amplitude_index][index]  = test_extraction_static_monochrome_LCDM[0].psi_squared
        master_term_number_array_CDM[amplitude_index][index] = test_extraction_static_monochrome_LCDM[0].sizes["systematics"]
        master_term_number_array_CDM_signal[amplitude_index][index]  = test_extraction_static_monochrome_LCDM[0].sizes["signal"]
        master_evidence_array_CDM[amplitude_index][index]= test_extraction_static_monochrome_LCDM[0].log_evidence


    plt.xlabel(r"$\Delta$ ln Z")
    plt.ylabel("counts")
    plt.title(f"Noise level of {N_antenna}: Standard Deviation of {(master_evidence_array-master_evidence_array_CDM).std():.2f} and mean of {(master_evidence_array-master_evidence_array_CDM).mean():.2f}")
    f = h5py.File(f"/home/dbarker7752/21_cm_group/Exotic_Models_Data/expected_distro_vs_Lambda_extract_{N_antenna}ants_{constraint}_Bayesian_{N_extractions}_amplitude_gridsearch","w")
    f.create_dataset("chi squared",data=master_chi_squared_array)
    f.create_dataset("psi squared",data=master_psi_squared_array)
    f.create_dataset("term number",data=master_term_number_array)
    f.create_dataset("Bayesian evidence",data=master_evidence_array)
    f.create_dataset("chi squared CDM",data=master_chi_squared_array_CDM)
    f.create_dataset("psi squared CDM",data=master_psi_squared_array_CDM)
    f.create_dataset("foreground term number CDM",data=master_term_number_array_CDM)
    f.create_dataset("signal term number CDM",data=master_term_number_array_CDM_signal)
    f.create_dataset("Bayesian evidence CDM",data=master_evidence_array_CDM)
    f.close()

for i,n in zip(amplitudes,range(len(master_evidence_array))):
    log_diff = master_evidence_array[n]-master_evidence_array_CDM[n]
    percent_detection = np.where(log_diff < -5)[0].shape[0]/N_extractions
    percent_detection_null = np.where(log_diff > 5)[0].shape[0]/N_extractions
    plt.scatter(i,percent_detection*100,color=CFG_color)
    print(f"CDM detection = {percent_detection*100}")
    print(f"Null detection = {percent_detection_null*100}")

plt.xlabel("amplitude multiplier")
plt.ylabel("percent detection")
plt.title(f"{constraint_label} Foreground Amplitude Grid Search")


In [ ]:
for i,n in zip(amplitudes,range(len(master_evidence_array))):
    log_diff = master_evidence_array[n]-master_evidence_array_CDM[n]
    percent_detection = np.where(log_diff < -5)[0].shape[0]/N_extractions
    # if percent_detection.shape[0] == 0:
    #     percent_detection = 0
    plt.scatter(i,percent_detection*100,color=CFG_color)
    print(percent_detection*100)
plt.xlabel("amplitude multiplier")
plt.ylabel("percent detection")
plt.title(f"{constraint_label} Foreground Amplitude Grid Search")

In [ ]:
for i,n in zip(amplitudes,range(len(master_evidence_array))):
    log_diff = master_evidence_array[n]-master_evidence_array_CDM[n]
    percent_detection = np.where(log_diff < -5)[0].shape[0]/N_extractions
    # if percent_detection.shape[0] == 0:
    #     percent_detection = 0
    plt.scatter(i,percent_detection*100,color=CFG_color)
plt.xlabel("amplitude multiplier")
plt.ylabel("percent detection")
plt.title("20% Foreground Amplitude Grid Search")

In [ ]:
## Emission region detects it as FG only but at the exact same sensitivity

for i,n in zip(amplitudes,range(len(master_evidence_array))):
    log_diff = master_evidence_array[n]-master_evidence_array_CDM[n]
    percent_detection = np.where(log_diff < -5)[0].shape[0]/N_extractions
    percent_detection_null = np.where(log_diff > 5)[0].shape[0]/N_extractions
    plt.scatter(i,percent_detection_null*100,color=CFG_color)
    print(f"CDM detection = {percent_detection*100}")
    print(f"Null detection = {percent_detection_null*100}")

plt.xlabel("amplitude multiplier")
plt.ylabel("percent detection")
plt.title(f"{constraint_label} Foreground Amplitude Grid Search")

#### 1% Foreground

In [ ]:

### User Variables ###

constraint_value = 0.0025
constraint_label = "1%"
# constraints = np.array([0.00000025])
N_antenna = 10
dnu=1e6
dt=100*3600
N_extractions = 10000
N_TS = 10000   # Number of curves in the training set
amplitudes = np.array([1.3])

evidence_cutoff = 0

### Code Backend ###
plot=False
verbose = False
n_LSTs = 10 # this should match the number of foreground_training_sets you've made
## Spectral Index Map Variable
n_regions = 8   # number of regions that are created for your multi-region model
temp1 = 20     # lower frequency that is used for the temperature for the spectral index of the spectral index map
temp2 = 50     # upper frequency
f_low = temp1   # Only concerns the residual maps that are displayed through spectral_index_map_visualizations (not important for this)
f_high = temp2  # Only concerns the residual maps that are displayed through spectral_index_map_visualizations (not important for this)
NSIDE = 32
## Monochromatic Beam Variables.
N_beam = 1   # Number of beams to create. I'm not varying the beam right now, but this may be helpful later.
low_std = 0.5
high_std = low_std
## Synchrotron Foreground Variables
ylim = (-0.07,0.03)

quantity = AttributeQuantity("log_evidence")



constraint=str(constraint_value)[2:]

parameter_variation = [0,constraint_value,0]
sky_maps = master_foreground_array
sky_maps_truncated = sky_maps[:,4:49,:]  # the trucated sky maps that map the frequency range of frequencies (though it may not match the number of bins)
frequency_range_truncated = np.arange(5,50)  # Truncated frequency range that matches the sky map frequencies
# Pylinex Extraction Variables
manual_systematics_terms = 0
manual_signal_terms = 0
num_sig_vectors = 3 # maximum number of signal modes to use in the extractions
num_sys_vectors = 75    # maximum number of systematic modes to use in the extraction
num_basis_vectors = 75  # number of basis vectors to create for the root bases (make sure its as big as the largest num_sys or num_sig_vectors number)
## signal that is used as the input to the observation data: ##
parameter_array = np.array([[omR0,0],[omM0h2,0],[omK0,0],[omL0,0],[omB0h2,0/h**2],[H0,0]])

##  ##
signal_training_set = lambdaCDM_training_set[:,16::4]  ### indexing is needed to get it into the correct shape for the extraction. Training set used in the pylinex fit
IC = "DIC"

master_chi_squared_array = np.zeros((len(amplitudes),N_extractions))
master_psi_squared_array = np.zeros((len(amplitudes),N_extractions))
master_term_number_array =np.zeros((len(amplitudes),N_extractions))
master_evidence_array = np.zeros((len(amplitudes),N_extractions))
master_term_number_array_signal =np.zeros((len(amplitudes),N_extractions))

# array for the Lambda CDM extraction (not the null signal, that's the ones above) 
master_chi_squared_array_CDM = np.zeros((len(amplitudes),N_extractions))
master_psi_squared_array_CDM = np.zeros((len(amplitudes),N_extractions))
master_term_number_array_CDM = np.zeros((len(amplitudes),N_extractions))
master_evidence_array_CDM = np.zeros((len(amplitudes),N_extractions))
master_term_number_array_CDM_signal = np.zeros((len(amplitudes),N_extractions))


#Step 1: Create the spectral index map that will govern the B_values of the beam (since they govern the indices of the regions).
# NPIX = hp.nside2npix(NSIDE)
# spectral_maps = np.zeros((n_LSTs,NPIX))
# for l in range(n_LSTs):
#     spectral_map = py21cmsig.spectral_index_map(f_low,f_high,n_regions,temp1,temp2,NSIDE,ULSA_direction_32 = master_foreground_array[l])
#     spectral_maps[l] = spectral_map

# f = h5py.File("/home/dbarker7752/21_cm_group/Exotic_Models_Data/spectral_index_map_10to50MHz_32bit","w")
# f.create_dataset("spectral maps",data=spectral_maps)
# f.close()

if NSIDE != 32:
    raise ValueError ("The saved spectral maps are only 32 bit right now.")
hdf5 = h5py.File("/home/dbarker7752/21_cm_group/Exotic_Models_Data/spectral_index_map_10to50MHz_32bit","r")
spectral_maps = np.array(hdf5["spectral maps"])   # I'v visually varified that these make sense

BTS_curves_monochromaticbeam = np.zeros((n_LSTs,len(frequencies)))
# # Step 2: Create the monochromatic beam
for l in np.arange(n_LSTs):
    BTS_curves_monochromaticbeam_raw = (py21cmsig.gaussian_beams(frequency_range_truncated,low_std,resolution=NSIDE,monochromatic_mode=True)[0]*sky_maps_truncated).sum(axis=2)
    BTS_interpolator = scipy.interpolate.CubicSpline(frequency_range_truncated,BTS_curves_monochromaticbeam_raw[l])
    BTS_curves_monochromaticbeam[l] = BTS_interpolator(frequencies)

# creates the B values
N=1
monochromatic_gaussian_B_values = np.zeros((n_LSTs,N,len(frequencies),n_regions))
gaussian_beam_element = py21cmsig.gaussian_beams(frequencies,0.5,NSIDE,monochromatic_mode=True)

for l in range(n_LSTs):
    patch=perses.models.PatchyForegroundModel(frequencies,spectral_maps[l],n_regions) # define the regional patches
    region_indices = patch.foreground_pixel_indices_by_region_dictionary
    for r in range(n_regions):
        monochromatic_gaussian_B_values[l,0,:,r] = gaussian_beam_element[:,region_indices[r]].sum(axis=1)
# monochromatic_gaussian_B_values = py21cmsig.create_monochromatic_gaussian_beam_training_set(frequencies,N_beam,[low_std,high_std],n_LSTs,n_regions,NSIDE,spectral_maps,plot_extremes = True)[0]
monochromatic_gaussian_parameters = np.array([low_std])
# if len(frequencies) != monochromatic_gaussian_B_values[0][0].shape[0]:
#     raise ValueError ("Your loaded B_values do not have the same length as your frequencies. This will cause problems later")

# # Step 3: Create the beam-weighted foreground training set:
# # Step 3a: Create the BTS_curves to input to the foreground function:

# simulated_foreground_array sets the "true sky" observations that will be compared to the training sets for the extraction. It's your data, basically. 
parameter_variation_true = [0,0,0]   
simulated_foreground_array_10LSTs = np.zeros((n_LSTs,len(frequencies)))
for l in range(n_LSTs):
    simulated_foreground_array_10LSTs[l,:] = py21cmsig.synchrotron_foreground_updated_given_Bvalues(1,n_regions,frequencies,spectral_maps[l],sky_maps[l],parameter_variation_true,\
                                                                                    np.array([monochromatic_gaussian_B_values[l,0]]),np.array([low_std]),\
                                                                                    np.array([BTS_curves_monochromaticbeam[l]]))[0]

# The systematics training set itself
monochro_beam_TS_10LSTs = np.zeros((N_TS,n_LSTs,len(frequencies)))
for l in range(n_LSTs):
    monochro_beam_TS_10LSTs[:,l,:] = py21cmsig.synchrotron_foreground_updated_given_Bvalues(N_TS,n_regions,frequencies,spectral_maps[l],sky_maps[l],parameter_variation,\
                                                                                    np.array([monochromatic_gaussian_B_values[l,0]]),np.array([monochromatic_gaussian_parameters[0]]),\
                                                                                    np.array([BTS_curves_monochromaticbeam[l]]))[0]



    #Pylinex Extractions



    extraction_dict = {}

    ### Pylinex part (doing it manually since we are only using the foreground for this) ###
# for amplitude_index in tqdm(np.arange(len(amplitudes))):
for amplitude_index in np.arange(len(amplitudes)):
    amplitude = amplitudes[amplitude_index]
    print(f"Beginning processing of {amplitude}x amplitude multiplier.")
    for index in tqdm(np.arange(N_extractions)):  
    
        input_signal = fiducial_signal*amplitude
        foreground = monochro_beam_TS_10LSTs[np.random.randint(N_TS)]
        static_monchrome_beam_simulation=py21cmsig.multi_spectra_simulation_run(frequencies,foreground,input_signal,N_antenna,dnu,dt)
        data=static_monchrome_beam_simulation[0]
        noise = static_monchrome_beam_simulation[5]
        signal = static_monchrome_beam_simulation[1]
        systematics = static_monchrome_beam_simulation[2]
        systematics_training_set = monochro_beam_TS_10LSTs
        flattened_systematics_training_set = np.reshape(systematics_training_set, (len(systematics_training_set), -1))
        temperatures_flat = data.flatten()
        noise_level_flat = noise.flatten()
        Ns = len(noise)      # grabs the number of correlated spectra you are using
        names = ["systematics"]        # names of your bases
        bases = [
        TrainedBasis(
            training_set=flattened_systematics_training_set,
            num_basis_vectors=num_basis_vectors,
            error=noise_level_flat,
            expander=NullExpander(),
            mean_translation=True
        )]
        basis_sum = BasisSum(names,bases)
        dimension = [{'systematics' : np.arange(1,num_sys_vectors)}]
        priors = {"systematics_prior" : bases[0].gaussian_prior} 
        meta_fitter = MetaFitter(basis_sum, temperatures_flat, noise_level_flat, quantity, quantity.name, *dimension,**priors,verbose=False)      # creates the meta_fitter object, which is an essential Pylinex object
        logZ_grid = meta_fitter["log_evidence"]

        best_indices = np.unravel_index(
            np.argmax(logZ_grid),
            logZ_grid.shape
        )

        fitter = meta_fitter.fitter_from_indices(best_indices)

        # print("Best term numbers:", fitter.sizes)
        # print("Maximum log evidence:", fitter.log_evidence)
                                                                    
        num_systematics_terms = fitter.sizes["systematics"]                                                
        # plot for the signal extraction
        pylinex_error = fitter.subbasis_channel_error("systematics")
        radiometer_error = noise[0]
        # plt.plot((fitter.subbasis_channel_mean("systematics")-temperatures_flat)/noise_level_flat,label = "residual fit mean",color="blue")
        # print(f"systematic terms used: {num_systematics_terms}")
        # print(f"reduced chi squared: {fitter.reduced_chi_squared}")
        # print(f"reduced psi squared: {fitter.psi_squared}")
        chi_squared = fitter.reduced_chi_squared
        psi_squared = fitter.psi_squared
        term_number = num_systematics_terms

        master_chi_squared_array[amplitude_index][index] = chi_squared
        master_psi_squared_array[amplitude_index][index] = psi_squared
        master_term_number_array[amplitude_index][index]= term_number   
        master_evidence_array[amplitude_index][index]= fitter.log_evidence

            ##### Lambda CDM extractions
        data=static_monchrome_beam_simulation[0]
        noise = static_monchrome_beam_simulation[5]
        signal = static_monchrome_beam_simulation[1]
        systematics = static_monchrome_beam_simulation[2]
        test_extraction_static_monochrome_LCDM = pylinex_extraction(systematics_training_set,signal_training_set,data,noise,signal,frequencies,IC="log_evidence",verbose=verbose,plot=plot,plot_residual=False,num_basis_vectors=num_basis_vectors,num_sys_vectors = num_sys_vectors, \
                        num_sig_vectors = num_sig_vectors, title = "Pylinex Extraction", ignore_IC = False,ylim=None,man_sys_terms=manual_systematics_terms,man_sig_terms=manual_signal_terms,man_sys_terms_lower=0,man_sig_terms_lower=0,multi_spectra=True,priors=True,covariance_expansion_factor=1\
                            ,evidence_cutoff=evidence_cutoff)



        master_chi_squared_array_CDM[amplitude_index][index] = test_extraction_static_monochrome_LCDM[0].reduced_chi_squared
        master_psi_squared_array_CDM[amplitude_index][index]  = test_extraction_static_monochrome_LCDM[0].psi_squared
        master_term_number_array_CDM[amplitude_index][index] = test_extraction_static_monochrome_LCDM[0].sizes["systematics"]
        master_term_number_array_CDM_signal[amplitude_index][index]  = test_extraction_static_monochrome_LCDM[0].sizes["signal"]
        master_evidence_array_CDM[amplitude_index][index]= test_extraction_static_monochrome_LCDM[0].log_evidence


    plt.xlabel(r"$\Delta$ ln Z")
    plt.ylabel("counts")
    plt.title(f"Noise level of {N_antenna}: Standard Deviation of {(master_evidence_array-master_evidence_array_CDM).std():.2f} and mean of {(master_evidence_array-master_evidence_array_CDM).mean():.2f}")
    f = h5py.File(f"/home/dbarker7752/21_cm_group/Exotic_Models_Data/expected_distro_vs_Lambda_extract_{N_antenna}ants_{constraint}_Bayesian_{N_extractions}_amplitude_gridsearch","w")
    f.create_dataset("chi squared",data=master_chi_squared_array)
    f.create_dataset("psi squared",data=master_psi_squared_array)
    f.create_dataset("term number",data=master_term_number_array)
    f.create_dataset("Bayesian evidence",data=master_evidence_array)
    f.create_dataset("chi squared CDM",data=master_chi_squared_array_CDM)
    f.create_dataset("psi squared CDM",data=master_psi_squared_array_CDM)
    f.create_dataset("foreground term number CDM",data=master_term_number_array_CDM)
    f.create_dataset("signal term number CDM",data=master_term_number_array_CDM_signal)
    f.create_dataset("Bayesian evidence CDM",data=master_evidence_array_CDM)
    f.close()

for i,n in zip(amplitudes,range(len(master_evidence_array))):
    log_diff = master_evidence_array[n]-master_evidence_array_CDM[n]
    percent_detection = np.where(log_diff < -5)[0].shape[0]/N_extractions
    percent_detection_null = np.where(log_diff > 5)[0].shape[0]/N_extractions
    plt.scatter(i,percent_detection*100,color=OFG_color)
    print(f"CDM detection = {percent_detection*100}")
    print(f"Null detection = {percent_detection_null*100}")
    

plt.xlabel("amplitude multiplier")
plt.ylabel("percent detection")
plt.title(f"{constraint_label} Foreground Amplitude Grid Search")


In [ ]:
### NULL ###
### User Variables ###

constraint_value = 0.0025
constraint_label = "1%"
# constraints = np.array([0.00000025])
N_antenna = 10
dnu=1e6
dt=100*3600
N_extractions = 1000
N_TS = 10000   # Number of curves in the training set
amplitudes = np.arange(0.4,1,0.1)

evidence_cutoff = 0

### Code Backend ###
plot=False
verbose = False
n_LSTs = 10 # this should match the number of foreground_training_sets you've made
## Spectral Index Map Variable
n_regions = 8   # number of regions that are created for your multi-region model
temp1 = 20     # lower frequency that is used for the temperature for the spectral index of the spectral index map
temp2 = 50     # upper frequency
f_low = temp1   # Only concerns the residual maps that are displayed through spectral_index_map_visualizations (not important for this)
f_high = temp2  # Only concerns the residual maps that are displayed through spectral_index_map_visualizations (not important for this)
NSIDE = 32
## Monochromatic Beam Variables.
N_beam = 1   # Number of beams to create. I'm not varying the beam right now, but this may be helpful later.
low_std = 0.5
high_std = low_std
## Synchrotron Foreground Variables
ylim = (-0.07,0.03)

quantity = AttributeQuantity("log_evidence")



constraint=str(constraint_value)[2:]

parameter_variation = [0,constraint_value,0]
sky_maps = master_foreground_array
sky_maps_truncated = sky_maps[:,4:49,:]  # the trucated sky maps that map the frequency range of frequencies (though it may not match the number of bins)
frequency_range_truncated = np.arange(5,50)  # Truncated frequency range that matches the sky map frequencies
# Pylinex Extraction Variables
manual_systematics_terms = 0
manual_signal_terms = 0
num_sig_vectors = 3 # maximum number of signal modes to use in the extractions
num_sys_vectors = 75    # maximum number of systematic modes to use in the extraction
num_basis_vectors = 75  # number of basis vectors to create for the root bases (make sure its as big as the largest num_sys or num_sig_vectors number)
## signal that is used as the input to the observation data: ##
parameter_array = np.array([[omR0,0],[omM0h2,0],[omK0,0],[omL0,0],[omB0h2,0/h**2],[H0,0]])

##  ##
signal_training_set = lambdaCDM_training_set[:,16::4]  ### indexing is needed to get it into the correct shape for the extraction. Training set used in the pylinex fit
IC = "DIC"

master_chi_squared_array = np.zeros((len(amplitudes),N_extractions))
master_psi_squared_array = np.zeros((len(amplitudes),N_extractions))
master_term_number_array =np.zeros((len(amplitudes),N_extractions))
master_evidence_array = np.zeros((len(amplitudes),N_extractions))
master_term_number_array_signal =np.zeros((len(amplitudes),N_extractions))

# array for the Lambda CDM extraction (not the null signal, that's the ones above) 
master_chi_squared_array_CDM = np.zeros((len(amplitudes),N_extractions))
master_psi_squared_array_CDM = np.zeros((len(amplitudes),N_extractions))
master_term_number_array_CDM = np.zeros((len(amplitudes),N_extractions))
master_evidence_array_CDM = np.zeros((len(amplitudes),N_extractions))
master_term_number_array_CDM_signal = np.zeros((len(amplitudes),N_extractions))


#Step 1: Create the spectral index map that will govern the B_values of the beam (since they govern the indices of the regions).
# NPIX = hp.nside2npix(NSIDE)
# spectral_maps = np.zeros((n_LSTs,NPIX))
# for l in range(n_LSTs):
#     spectral_map = py21cmsig.spectral_index_map(f_low,f_high,n_regions,temp1,temp2,NSIDE,ULSA_direction_32 = master_foreground_array[l])
#     spectral_maps[l] = spectral_map

# f = h5py.File("/home/dbarker7752/21_cm_group/Exotic_Models_Data/spectral_index_map_10to50MHz_32bit","w")
# f.create_dataset("spectral maps",data=spectral_maps)
# f.close()

if NSIDE != 32:
    raise ValueError ("The saved spectral maps are only 32 bit right now.")
hdf5 = h5py.File("/home/dbarker7752/21_cm_group/Exotic_Models_Data/spectral_index_map_10to50MHz_32bit","r")
spectral_maps = np.array(hdf5["spectral maps"])   # I'v visually varified that these make sense

BTS_curves_monochromaticbeam = np.zeros((n_LSTs,len(frequencies)))
# # Step 2: Create the monochromatic beam
for l in np.arange(n_LSTs):
    BTS_curves_monochromaticbeam_raw = (py21cmsig.gaussian_beams(frequency_range_truncated,low_std,resolution=NSIDE,monochromatic_mode=True)[0]*sky_maps_truncated).sum(axis=2)
    BTS_interpolator = scipy.interpolate.CubicSpline(frequency_range_truncated,BTS_curves_monochromaticbeam_raw[l])
    BTS_curves_monochromaticbeam[l] = BTS_interpolator(frequencies)

# creates the B values
N=1
monochromatic_gaussian_B_values = np.zeros((n_LSTs,N,len(frequencies),n_regions))
gaussian_beam_element = py21cmsig.gaussian_beams(frequencies,0.5,NSIDE,monochromatic_mode=True)

for l in range(n_LSTs):
    patch=perses.models.PatchyForegroundModel(frequencies,spectral_maps[l],n_regions) # define the regional patches
    region_indices = patch.foreground_pixel_indices_by_region_dictionary
    for r in range(n_regions):
        monochromatic_gaussian_B_values[l,0,:,r] = gaussian_beam_element[:,region_indices[r]].sum(axis=1)
# monochromatic_gaussian_B_values = py21cmsig.create_monochromatic_gaussian_beam_training_set(frequencies,N_beam,[low_std,high_std],n_LSTs,n_regions,NSIDE,spectral_maps,plot_extremes = True)[0]
monochromatic_gaussian_parameters = np.array([low_std])
# if len(frequencies) != monochromatic_gaussian_B_values[0][0].shape[0]:
#     raise ValueError ("Your loaded B_values do not have the same length as your frequencies. This will cause problems later")

# # Step 3: Create the beam-weighted foreground training set:
# # Step 3a: Create the BTS_curves to input to the foreground function:

# simulated_foreground_array sets the "true sky" observations that will be compared to the training sets for the extraction. It's your data, basically. 
parameter_variation_true = [0,0,0]   
simulated_foreground_array_10LSTs = np.zeros((n_LSTs,len(frequencies)))
for l in range(n_LSTs):
    simulated_foreground_array_10LSTs[l,:] = py21cmsig.synchrotron_foreground_updated_given_Bvalues(1,n_regions,frequencies,spectral_maps[l],sky_maps[l],parameter_variation_true,\
                                                                                    np.array([monochromatic_gaussian_B_values[l,0]]),np.array([low_std]),\
                                                                                    np.array([BTS_curves_monochromaticbeam[l]]))[0]

# The systematics training set itself
monochro_beam_TS_10LSTs = np.zeros((N_TS,n_LSTs,len(frequencies)))
for l in range(n_LSTs):
    monochro_beam_TS_10LSTs[:,l,:] = py21cmsig.synchrotron_foreground_updated_given_Bvalues(N_TS,n_regions,frequencies,spectral_maps[l],sky_maps[l],parameter_variation,\
                                                                                    np.array([monochromatic_gaussian_B_values[l,0]]),np.array([monochromatic_gaussian_parameters[0]]),\
                                                                                    np.array([BTS_curves_monochromaticbeam[l]]))[0]



    #Pylinex Extractions



    extraction_dict = {}

    ### Pylinex part (doing it manually since we are only using the foreground for this) ###
# for amplitude_index in tqdm(np.arange(len(amplitudes))):
for amplitude_index in np.arange(len(amplitudes)):
    amplitude = amplitudes[amplitude_index]
    print(f"Beginning processing of {amplitude}x amplitude multiplier.")
    for index in tqdm(np.arange(N_extractions)):  
    
        input_signal = -fiducial_signal*amplitude
        foreground = monochro_beam_TS_10LSTs[np.random.randint(N_TS)]
        static_monchrome_beam_simulation=py21cmsig.multi_spectra_simulation_run(frequencies,foreground,input_signal,N_antenna,dnu,dt)
        data=static_monchrome_beam_simulation[0]
        noise = static_monchrome_beam_simulation[5]
        signal = static_monchrome_beam_simulation[1]
        systematics = static_monchrome_beam_simulation[2]
        systematics_training_set = monochro_beam_TS_10LSTs
        flattened_systematics_training_set = np.reshape(systematics_training_set, (len(systematics_training_set), -1))
        temperatures_flat = data.flatten()
        noise_level_flat = noise.flatten()
        Ns = len(noise)      # grabs the number of correlated spectra you are using
        names = ["systematics"]        # names of your bases
        bases = [
        TrainedBasis(
            training_set=flattened_systematics_training_set,
            num_basis_vectors=num_basis_vectors,
            error=noise_level_flat,
            expander=NullExpander(),
            mean_translation=True
        )]
        basis_sum = BasisSum(names,bases)
        dimension = [{'systematics' : np.arange(1,num_sys_vectors)}]
        priors = {"systematics_prior" : bases[0].gaussian_prior} 
        meta_fitter = MetaFitter(basis_sum, temperatures_flat, noise_level_flat, quantity, quantity.name, *dimension,**priors,verbose=False)      # creates the meta_fitter object, which is an essential Pylinex object
        logZ_grid = meta_fitter["log_evidence"]

        best_indices = np.unravel_index(
            np.argmax(logZ_grid),
            logZ_grid.shape
        )

        fitter = meta_fitter.fitter_from_indices(best_indices)

        # print("Best term numbers:", fitter.sizes)
        # print("Maximum log evidence:", fitter.log_evidence)
                                                                    
        num_systematics_terms = fitter.sizes["systematics"]                                                
        # plot for the signal extraction
        pylinex_error = fitter.subbasis_channel_error("systematics")
        radiometer_error = noise[0]
        # plt.plot((fitter.subbasis_channel_mean("systematics")-temperatures_flat)/noise_level_flat,label = "residual fit mean",color="blue")
        # print(f"systematic terms used: {num_systematics_terms}")
        # print(f"reduced chi squared: {fitter.reduced_chi_squared}")
        # print(f"reduced psi squared: {fitter.psi_squared}")
        chi_squared = fitter.reduced_chi_squared
        psi_squared = fitter.psi_squared
        term_number = num_systematics_terms

        master_chi_squared_array[amplitude_index][index] = chi_squared
        master_psi_squared_array[amplitude_index][index] = psi_squared
        master_term_number_array[amplitude_index][index]= term_number   
        master_evidence_array[amplitude_index][index]= fitter.log_evidence

            ##### Lambda CDM extractions
        data=static_monchrome_beam_simulation[0]
        noise = static_monchrome_beam_simulation[5]
        signal = static_monchrome_beam_simulation[1]
        systematics = static_monchrome_beam_simulation[2]
        test_extraction_static_monochrome_LCDM = pylinex_extraction(systematics_training_set,signal_training_set,data,noise,signal,frequencies,IC="log_evidence",verbose=verbose,plot=plot,plot_residual=False,num_basis_vectors=num_basis_vectors,num_sys_vectors = num_sys_vectors, \
                        num_sig_vectors = num_sig_vectors, title = "Pylinex Extraction", ignore_IC = False,ylim=None,man_sys_terms=manual_systematics_terms,man_sig_terms=manual_signal_terms,man_sys_terms_lower=0,man_sig_terms_lower=0,multi_spectra=True,priors=True,covariance_expansion_factor=1\
                            ,evidence_cutoff=evidence_cutoff)



        master_chi_squared_array_CDM[amplitude_index][index] = test_extraction_static_monochrome_LCDM[0].reduced_chi_squared
        master_psi_squared_array_CDM[amplitude_index][index]  = test_extraction_static_monochrome_LCDM[0].psi_squared
        master_term_number_array_CDM[amplitude_index][index] = test_extraction_static_monochrome_LCDM[0].sizes["systematics"]
        master_term_number_array_CDM_signal[amplitude_index][index]  = test_extraction_static_monochrome_LCDM[0].sizes["signal"]
        master_evidence_array_CDM[amplitude_index][index]= test_extraction_static_monochrome_LCDM[0].log_evidence


    plt.xlabel(r"$\Delta$ ln Z")
    plt.ylabel("counts")
    plt.title(f"Noise level of {N_antenna}: Standard Deviation of {(master_evidence_array-master_evidence_array_CDM).std():.2f} and mean of {(master_evidence_array-master_evidence_array_CDM).mean():.2f}")
    f = h5py.File(f"/home/dbarker7752/21_cm_group/Exotic_Models_Data/expected_distro_vs_Lambda_extract_{N_antenna}ants_{constraint}_Bayesian_{N_extractions}_amplitude_gridsearch_top","w")
    f.create_dataset("chi squared",data=master_chi_squared_array)
    f.create_dataset("psi squared",data=master_psi_squared_array)
    f.create_dataset("term number",data=master_term_number_array)
    f.create_dataset("Bayesian evidence",data=master_evidence_array)
    f.create_dataset("chi squared CDM",data=master_chi_squared_array_CDM)
    f.create_dataset("psi squared CDM",data=master_psi_squared_array_CDM)
    f.create_dataset("foreground term number CDM",data=master_term_number_array_CDM)
    f.create_dataset("signal term number CDM",data=master_term_number_array_CDM_signal)
    f.create_dataset("Bayesian evidence CDM",data=master_evidence_array_CDM)
    f.close()

for i,n in zip(amplitudes,range(len(master_evidence_array))):
    log_diff = master_evidence_array[n]-master_evidence_array_CDM[n]
    percent_detection = np.where(log_diff < -5)[0].shape[0]/N_extractions
    percent_detection_null = np.where(log_diff > 5)[0].shape[0]/N_extractions
    plt.scatter(i,percent_detection_null*100,color=OFG_color)
    print(f"CDM detection = {percent_detection*100}")
    print(f"Null detection = {percent_detection_null*100}")

plt.xlabel("amplitude multiplier")
plt.ylabel("percent detection")
plt.title(f"{constraint_label} Foreground Amplitude Grid Search")


In [ ]:

N_antenna = 10
constraint = "0025"
N_extractions = "100"
f = h5py.File(f"/home/dbarker7752/21_cm_group/Exotic_Models_Data/expected_distro_vs_Lambda_extract_{N_antenna}ants_{constraint}_Bayesian_{N_extractions}_amplitude_gridsearch","r")
master_evidence_array = np.array(f["Bayesian evidence"])
master_evidence_array_CDM = np.array(f["Bayesian evidence CDM"])
amplitudes = np.arange(0,1.5,0.1)
N_extractions = 100

for i,n in enumerate(amplitudes):
    log_diff = master_evidence_array[i]-master_evidence_array_CDM[i]
    percent_detection = np.where(log_diff < -5)[0].shape[0]/N_extractions
    percent_detection_null = np.where(log_diff > 5)[0].shape[0]/N_extractions
    plt.scatter(n,percent_detection_null*100,color=OFG_color)
    print(f"CDM detection = {percent_detection*100}")
    print(f"Null detection = {percent_detection_null*100}")

#### 0.1 % Foreground

In [ ]:

### User Variables ###

constraint_value = 0.00025
constraint_label = "0.1%"
# constraints = np.array([0.00000025])
N_antenna = 10
dnu=1e6
dt=100*3600
N_extractions = 10000
N_TS = 10000   # Number of curves in the training set
amplitudes = np.array([1.3])

evidence_cutoff = 0

### Code Backend ###
plot=False
verbose = False
n_LSTs = 10 # this should match the number of foreground_training_sets you've made
## Spectral Index Map Variable
n_regions = 8   # number of regions that are created for your multi-region model
temp1 = 20     # lower frequency that is used for the temperature for the spectral index of the spectral index map
temp2 = 50     # upper frequency
f_low = temp1   # Only concerns the residual maps that are displayed through spectral_index_map_visualizations (not important for this)
f_high = temp2  # Only concerns the residual maps that are displayed through spectral_index_map_visualizations (not important for this)
NSIDE = 32
## Monochromatic Beam Variables.
N_beam = 1   # Number of beams to create. I'm not varying the beam right now, but this may be helpful later.
low_std = 0.5
high_std = low_std
## Synchrotron Foreground Variables
ylim = (-0.07,0.03)

quantity = AttributeQuantity("log_evidence")



constraint=str(constraint_value)[2:]

parameter_variation = [0,constraint_value,0]
sky_maps = master_foreground_array
sky_maps_truncated = sky_maps[:,4:49,:]  # the trucated sky maps that map the frequency range of frequencies (though it may not match the number of bins)
frequency_range_truncated = np.arange(5,50)  # Truncated frequency range that matches the sky map frequencies
# Pylinex Extraction Variables
manual_systematics_terms = 0
manual_signal_terms = 0
num_sig_vectors = 3 # maximum number of signal modes to use in the extractions
num_sys_vectors = 75    # maximum number of systematic modes to use in the extraction
num_basis_vectors = 75  # number of basis vectors to create for the root bases (make sure its as big as the largest num_sys or num_sig_vectors number)
## signal that is used as the input to the observation data: ##
parameter_array = np.array([[omR0,0],[omM0h2,0],[omK0,0],[omL0,0],[omB0h2,0/h**2],[H0,0]])

##  ##
signal_training_set = lambdaCDM_training_set[:,16::4]  ### indexing is needed to get it into the correct shape for the extraction. Training set used in the pylinex fit
IC = "DIC"

master_chi_squared_array = np.zeros((len(amplitudes),N_extractions))
master_psi_squared_array = np.zeros((len(amplitudes),N_extractions))
master_term_number_array =np.zeros((len(amplitudes),N_extractions))
master_evidence_array = np.zeros((len(amplitudes),N_extractions))
master_term_number_array_signal =np.zeros((len(amplitudes),N_extractions))

# array for the Lambda CDM extraction (not the null signal, that's the ones above) 
master_chi_squared_array_CDM = np.zeros((len(amplitudes),N_extractions))
master_psi_squared_array_CDM = np.zeros((len(amplitudes),N_extractions))
master_term_number_array_CDM = np.zeros((len(amplitudes),N_extractions))
master_evidence_array_CDM = np.zeros((len(amplitudes),N_extractions))
master_term_number_array_CDM_signal = np.zeros((len(amplitudes),N_extractions))


#Step 1: Create the spectral index map that will govern the B_values of the beam (since they govern the indices of the regions).
# NPIX = hp.nside2npix(NSIDE)
# spectral_maps = np.zeros((n_LSTs,NPIX))
# for l in range(n_LSTs):
#     spectral_map = py21cmsig.spectral_index_map(f_low,f_high,n_regions,temp1,temp2,NSIDE,ULSA_direction_32 = master_foreground_array[l])
#     spectral_maps[l] = spectral_map

# f = h5py.File("/home/dbarker7752/21_cm_group/Exotic_Models_Data/spectral_index_map_10to50MHz_32bit","w")
# f.create_dataset("spectral maps",data=spectral_maps)
# f.close()

if NSIDE != 32:
    raise ValueError ("The saved spectral maps are only 32 bit right now.")
hdf5 = h5py.File("/home/dbarker7752/21_cm_group/Exotic_Models_Data/spectral_index_map_10to50MHz_32bit","r")
spectral_maps = np.array(hdf5["spectral maps"])   # I'v visually varified that these make sense

BTS_curves_monochromaticbeam = np.zeros((n_LSTs,len(frequencies)))
# # Step 2: Create the monochromatic beam
for l in np.arange(n_LSTs):
    BTS_curves_monochromaticbeam_raw = (py21cmsig.gaussian_beams(frequency_range_truncated,low_std,resolution=NSIDE,monochromatic_mode=True)[0]*sky_maps_truncated).sum(axis=2)
    BTS_interpolator = scipy.interpolate.CubicSpline(frequency_range_truncated,BTS_curves_monochromaticbeam_raw[l])
    BTS_curves_monochromaticbeam[l] = BTS_interpolator(frequencies)

# creates the B values
N=1
monochromatic_gaussian_B_values = np.zeros((n_LSTs,N,len(frequencies),n_regions))
gaussian_beam_element = py21cmsig.gaussian_beams(frequencies,0.5,NSIDE,monochromatic_mode=True)

for l in range(n_LSTs):
    patch=perses.models.PatchyForegroundModel(frequencies,spectral_maps[l],n_regions) # define the regional patches
    region_indices = patch.foreground_pixel_indices_by_region_dictionary
    for r in range(n_regions):
        monochromatic_gaussian_B_values[l,0,:,r] = gaussian_beam_element[:,region_indices[r]].sum(axis=1)
# monochromatic_gaussian_B_values = py21cmsig.create_monochromatic_gaussian_beam_training_set(frequencies,N_beam,[low_std,high_std],n_LSTs,n_regions,NSIDE,spectral_maps,plot_extremes = True)[0]
monochromatic_gaussian_parameters = np.array([low_std])
# if len(frequencies) != monochromatic_gaussian_B_values[0][0].shape[0]:
#     raise ValueError ("Your loaded B_values do not have the same length as your frequencies. This will cause problems later")

# # Step 3: Create the beam-weighted foreground training set:
# # Step 3a: Create the BTS_curves to input to the foreground function:

# simulated_foreground_array sets the "true sky" observations that will be compared to the training sets for the extraction. It's your data, basically. 
parameter_variation_true = [0,0,0]   
simulated_foreground_array_10LSTs = np.zeros((n_LSTs,len(frequencies)))
for l in range(n_LSTs):
    simulated_foreground_array_10LSTs[l,:] = py21cmsig.synchrotron_foreground_updated_given_Bvalues(1,n_regions,frequencies,spectral_maps[l],sky_maps[l],parameter_variation_true,\
                                                                                    np.array([monochromatic_gaussian_B_values[l,0]]),np.array([low_std]),\
                                                                                    np.array([BTS_curves_monochromaticbeam[l]]))[0]

# The systematics training set itself
monochro_beam_TS_10LSTs = np.zeros((N_TS,n_LSTs,len(frequencies)))
for l in range(n_LSTs):
    monochro_beam_TS_10LSTs[:,l,:] = py21cmsig.synchrotron_foreground_updated_given_Bvalues(N_TS,n_regions,frequencies,spectral_maps[l],sky_maps[l],parameter_variation,\
                                                                                    np.array([monochromatic_gaussian_B_values[l,0]]),np.array([monochromatic_gaussian_parameters[0]]),\
                                                                                    np.array([BTS_curves_monochromaticbeam[l]]))[0]



    #Pylinex Extractions



    extraction_dict = {}

    ### Pylinex part (doing it manually since we are only using the foreground for this) ###
# for amplitude_index in tqdm(np.arange(len(amplitudes))):
for amplitude_index in np.arange(len(amplitudes)):
    amplitude = amplitudes[amplitude_index]
    print(f"Beginning processing of {amplitude}x amplitude multiplier.")
    for index in tqdm(np.arange(N_extractions)):  
    
        input_signal = fiducial_signal*amplitude
        foreground = monochro_beam_TS_10LSTs[np.random.randint(N_TS)]
        static_monchrome_beam_simulation=py21cmsig.multi_spectra_simulation_run(frequencies,foreground,input_signal,N_antenna,dnu,dt)
        data=static_monchrome_beam_simulation[0]
        noise = static_monchrome_beam_simulation[5]
        signal = static_monchrome_beam_simulation[1]
        systematics = static_monchrome_beam_simulation[2]
        systematics_training_set = monochro_beam_TS_10LSTs
        flattened_systematics_training_set = np.reshape(systematics_training_set, (len(systematics_training_set), -1))
        temperatures_flat = data.flatten()
        noise_level_flat = noise.flatten()
        Ns = len(noise)      # grabs the number of correlated spectra you are using
        names = ["systematics"]        # names of your bases
        bases = [
        TrainedBasis(
            training_set=flattened_systematics_training_set,
            num_basis_vectors=num_basis_vectors,
            error=noise_level_flat,
            expander=NullExpander(),
            mean_translation=True
        )]
        basis_sum = BasisSum(names,bases)
        dimension = [{'systematics' : np.arange(1,num_sys_vectors)}]
        priors = {"systematics_prior" : bases[0].gaussian_prior} 
        meta_fitter = MetaFitter(basis_sum, temperatures_flat, noise_level_flat, quantity, quantity.name, *dimension,**priors,verbose=False)      # creates the meta_fitter object, which is an essential Pylinex object
        logZ_grid = meta_fitter["log_evidence"]

        best_indices = np.unravel_index(
            np.argmax(logZ_grid),
            logZ_grid.shape
        )

        fitter = meta_fitter.fitter_from_indices(best_indices)

        # print("Best term numbers:", fitter.sizes)
        # print("Maximum log evidence:", fitter.log_evidence)
                                                                    
        num_systematics_terms = fitter.sizes["systematics"]                                                
        # plot for the signal extraction
        pylinex_error = fitter.subbasis_channel_error("systematics")
        radiometer_error = noise[0]
        # plt.plot((fitter.subbasis_channel_mean("systematics")-temperatures_flat)/noise_level_flat,label = "residual fit mean",color="blue")
        # print(f"systematic terms used: {num_systematics_terms}")
        # print(f"reduced chi squared: {fitter.reduced_chi_squared}")
        # print(f"reduced psi squared: {fitter.psi_squared}")
        chi_squared = fitter.reduced_chi_squared
        psi_squared = fitter.psi_squared
        term_number = num_systematics_terms

        master_chi_squared_array[amplitude_index][index] = chi_squared
        master_psi_squared_array[amplitude_index][index] = psi_squared
        master_term_number_array[amplitude_index][index]= term_number   
        master_evidence_array[amplitude_index][index]= fitter.log_evidence

            ##### Lambda CDM extractions
        data=static_monchrome_beam_simulation[0]
        noise = static_monchrome_beam_simulation[5]
        signal = static_monchrome_beam_simulation[1]
        systematics = static_monchrome_beam_simulation[2]
        test_extraction_static_monochrome_LCDM = pylinex_extraction(systematics_training_set,signal_training_set,data,noise,signal,frequencies,IC="log_evidence",verbose=verbose,plot=plot,plot_residual=False,num_basis_vectors=num_basis_vectors,num_sys_vectors = num_sys_vectors, \
                        num_sig_vectors = num_sig_vectors, title = "Pylinex Extraction", ignore_IC = False,ylim=None,man_sys_terms=manual_systematics_terms,man_sig_terms=manual_signal_terms,man_sys_terms_lower=0,man_sig_terms_lower=0,multi_spectra=True,priors=True,covariance_expansion_factor=1\
                            ,evidence_cutoff=evidence_cutoff)



        master_chi_squared_array_CDM[amplitude_index][index] = test_extraction_static_monochrome_LCDM[0].reduced_chi_squared
        master_psi_squared_array_CDM[amplitude_index][index]  = test_extraction_static_monochrome_LCDM[0].psi_squared
        master_term_number_array_CDM[amplitude_index][index] = test_extraction_static_monochrome_LCDM[0].sizes["systematics"]
        master_term_number_array_CDM_signal[amplitude_index][index]  = test_extraction_static_monochrome_LCDM[0].sizes["signal"]
        master_evidence_array_CDM[amplitude_index][index]= test_extraction_static_monochrome_LCDM[0].log_evidence


    plt.xlabel(r"$\Delta$ ln Z")
    plt.ylabel("counts")
    plt.title(f"Noise level of {N_antenna}: Standard Deviation of {(master_evidence_array-master_evidence_array_CDM).std():.2f} and mean of {(master_evidence_array-master_evidence_array_CDM).mean():.2f}")
    f = h5py.File(f"/home/dbarker7752/21_cm_group/Exotic_Models_Data/expected_distro_vs_Lambda_extract_{N_antenna}ants_{constraint}_Bayesian_{N_extractions}_amplitude_gridsearch","w")
    f.create_dataset("chi squared",data=master_chi_squared_array)
    f.create_dataset("psi squared",data=master_psi_squared_array)
    f.create_dataset("term number",data=master_term_number_array)
    f.create_dataset("Bayesian evidence",data=master_evidence_array)
    f.create_dataset("chi squared CDM",data=master_chi_squared_array_CDM)
    f.create_dataset("psi squared CDM",data=master_psi_squared_array_CDM)
    f.create_dataset("foreground term number CDM",data=master_term_number_array_CDM)
    f.create_dataset("signal term number CDM",data=master_term_number_array_CDM_signal)
    f.create_dataset("Bayesian evidence CDM",data=master_evidence_array_CDM)
    f.close()

for i,n in zip(amplitudes,range(len(master_evidence_array))):
    log_diff = master_evidence_array[n]-master_evidence_array_CDM[n]
    percent_detection = np.where(log_diff < -5)[0].shape[0]/N_extractions
    percent_detection_null = np.where(log_diff > 5)[0].shape[0]/N_extractions
    plt.scatter(i,percent_detection*100,color=OFG_color)
    print(f"CDM detection = {percent_detection*100}")
    print(f"Null detection = {percent_detection_null*100}")

plt.xlabel("amplitude multiplier")
plt.ylabel("percent detection")
plt.title(f"{constraint_label} Foreground Amplitude Grid Search")


In [ ]:

#### NULL ####

### User Variables ###

constraint_value = 0.00025
constraint_label = "0.1%"
# constraints = np.array([0.00000025])
N_antenna = 10
dnu=1e6
dt=100*3600
N_extractions = 1000
N_TS = 10000   # Number of curves in the training set
amplitudes = np.arange(0,1,0.1)

evidence_cutoff = 0

### Code Backend ###
plot=False
verbose = False
n_LSTs = 10 # this should match the number of foreground_training_sets you've made
## Spectral Index Map Variable
n_regions = 8   # number of regions that are created for your multi-region model
temp1 = 20     # lower frequency that is used for the temperature for the spectral index of the spectral index map
temp2 = 50     # upper frequency
f_low = temp1   # Only concerns the residual maps that are displayed through spectral_index_map_visualizations (not important for this)
f_high = temp2  # Only concerns the residual maps that are displayed through spectral_index_map_visualizations (not important for this)
NSIDE = 32
## Monochromatic Beam Variables.
N_beam = 1   # Number of beams to create. I'm not varying the beam right now, but this may be helpful later.
low_std = 0.5
high_std = low_std
## Synchrotron Foreground Variables
ylim = (-0.07,0.03)

quantity = AttributeQuantity("log_evidence")



constraint=str(constraint_value)[2:]

parameter_variation = [0,constraint_value,0]
sky_maps = master_foreground_array
sky_maps_truncated = sky_maps[:,4:49,:]  # the trucated sky maps that map the frequency range of frequencies (though it may not match the number of bins)
frequency_range_truncated = np.arange(5,50)  # Truncated frequency range that matches the sky map frequencies
# Pylinex Extraction Variables
manual_systematics_terms = 0
manual_signal_terms = 0
num_sig_vectors = 3 # maximum number of signal modes to use in the extractions
num_sys_vectors = 75    # maximum number of systematic modes to use in the extraction
num_basis_vectors = 75  # number of basis vectors to create for the root bases (make sure its as big as the largest num_sys or num_sig_vectors number)
## signal that is used as the input to the observation data: ##
parameter_array = np.array([[omR0,0],[omM0h2,0],[omK0,0],[omL0,0],[omB0h2,0/h**2],[H0,0]])

##  ##
signal_training_set = lambdaCDM_training_set[:,16::4]  ### indexing is needed to get it into the correct shape for the extraction. Training set used in the pylinex fit
IC = "DIC"

master_chi_squared_array = np.zeros((len(amplitudes),N_extractions))
master_psi_squared_array = np.zeros((len(amplitudes),N_extractions))
master_term_number_array =np.zeros((len(amplitudes),N_extractions))
master_evidence_array = np.zeros((len(amplitudes),N_extractions))
master_term_number_array_signal =np.zeros((len(amplitudes),N_extractions))

# array for the Lambda CDM extraction (not the null signal, that's the ones above) 
master_chi_squared_array_CDM = np.zeros((len(amplitudes),N_extractions))
master_psi_squared_array_CDM = np.zeros((len(amplitudes),N_extractions))
master_term_number_array_CDM = np.zeros((len(amplitudes),N_extractions))
master_evidence_array_CDM = np.zeros((len(amplitudes),N_extractions))
master_term_number_array_CDM_signal = np.zeros((len(amplitudes),N_extractions))


#Step 1: Create the spectral index map that will govern the B_values of the beam (since they govern the indices of the regions).
# NPIX = hp.nside2npix(NSIDE)
# spectral_maps = np.zeros((n_LSTs,NPIX))
# for l in range(n_LSTs):
#     spectral_map = py21cmsig.spectral_index_map(f_low,f_high,n_regions,temp1,temp2,NSIDE,ULSA_direction_32 = master_foreground_array[l])
#     spectral_maps[l] = spectral_map

# f = h5py.File("/home/dbarker7752/21_cm_group/Exotic_Models_Data/spectral_index_map_10to50MHz_32bit","w")
# f.create_dataset("spectral maps",data=spectral_maps)
# f.close()

if NSIDE != 32:
    raise ValueError ("The saved spectral maps are only 32 bit right now.")
hdf5 = h5py.File("/home/dbarker7752/21_cm_group/Exotic_Models_Data/spectral_index_map_10to50MHz_32bit","r")
spectral_maps = np.array(hdf5["spectral maps"])   # I'v visually varified that these make sense

BTS_curves_monochromaticbeam = np.zeros((n_LSTs,len(frequencies)))
# # Step 2: Create the monochromatic beam
for l in np.arange(n_LSTs):
    BTS_curves_monochromaticbeam_raw = (py21cmsig.gaussian_beams(frequency_range_truncated,low_std,resolution=NSIDE,monochromatic_mode=True)[0]*sky_maps_truncated).sum(axis=2)
    BTS_interpolator = scipy.interpolate.CubicSpline(frequency_range_truncated,BTS_curves_monochromaticbeam_raw[l])
    BTS_curves_monochromaticbeam[l] = BTS_interpolator(frequencies)

# creates the B values
N=1
monochromatic_gaussian_B_values = np.zeros((n_LSTs,N,len(frequencies),n_regions))
gaussian_beam_element = py21cmsig.gaussian_beams(frequencies,0.5,NSIDE,monochromatic_mode=True)

for l in range(n_LSTs):
    patch=perses.models.PatchyForegroundModel(frequencies,spectral_maps[l],n_regions) # define the regional patches
    region_indices = patch.foreground_pixel_indices_by_region_dictionary
    for r in range(n_regions):
        monochromatic_gaussian_B_values[l,0,:,r] = gaussian_beam_element[:,region_indices[r]].sum(axis=1)
# monochromatic_gaussian_B_values = py21cmsig.create_monochromatic_gaussian_beam_training_set(frequencies,N_beam,[low_std,high_std],n_LSTs,n_regions,NSIDE,spectral_maps,plot_extremes = True)[0]
monochromatic_gaussian_parameters = np.array([low_std])
# if len(frequencies) != monochromatic_gaussian_B_values[0][0].shape[0]:
#     raise ValueError ("Your loaded B_values do not have the same length as your frequencies. This will cause problems later")

# # Step 3: Create the beam-weighted foreground training set:
# # Step 3a: Create the BTS_curves to input to the foreground function:

# simulated_foreground_array sets the "true sky" observations that will be compared to the training sets for the extraction. It's your data, basically. 
parameter_variation_true = [0,0,0]   
simulated_foreground_array_10LSTs = np.zeros((n_LSTs,len(frequencies)))
for l in range(n_LSTs):
    simulated_foreground_array_10LSTs[l,:] = py21cmsig.synchrotron_foreground_updated_given_Bvalues(1,n_regions,frequencies,spectral_maps[l],sky_maps[l],parameter_variation_true,\
                                                                                    np.array([monochromatic_gaussian_B_values[l,0]]),np.array([low_std]),\
                                                                                    np.array([BTS_curves_monochromaticbeam[l]]))[0]

# The systematics training set itself
monochro_beam_TS_10LSTs = np.zeros((N_TS,n_LSTs,len(frequencies)))
for l in range(n_LSTs):
    monochro_beam_TS_10LSTs[:,l,:] = py21cmsig.synchrotron_foreground_updated_given_Bvalues(N_TS,n_regions,frequencies,spectral_maps[l],sky_maps[l],parameter_variation,\
                                                                                    np.array([monochromatic_gaussian_B_values[l,0]]),np.array([monochromatic_gaussian_parameters[0]]),\
                                                                                    np.array([BTS_curves_monochromaticbeam[l]]))[0]



    #Pylinex Extractions



    extraction_dict = {}

    ### Pylinex part (doing it manually since we are only using the foreground for this) ###
# for amplitude_index in tqdm(np.arange(len(amplitudes))):
for amplitude_index in np.arange(len(amplitudes)):
    amplitude = amplitudes[amplitude_index]
    print(f"Beginning processing of {amplitude}x amplitude multiplier.")
    for index in tqdm(np.arange(N_extractions)):  
    
        input_signal = -fiducial_signal*amplitude
        foreground = monochro_beam_TS_10LSTs[np.random.randint(N_TS)]
        static_monchrome_beam_simulation=py21cmsig.multi_spectra_simulation_run(frequencies,foreground,input_signal,N_antenna,dnu,dt)
        data=static_monchrome_beam_simulation[0]
        noise = static_monchrome_beam_simulation[5]
        signal = static_monchrome_beam_simulation[1]
        systematics = static_monchrome_beam_simulation[2]
        systematics_training_set = monochro_beam_TS_10LSTs
        flattened_systematics_training_set = np.reshape(systematics_training_set, (len(systematics_training_set), -1))
        temperatures_flat = data.flatten()
        noise_level_flat = noise.flatten()
        Ns = len(noise)      # grabs the number of correlated spectra you are using
        names = ["systematics"]        # names of your bases
        bases = [
        TrainedBasis(
            training_set=flattened_systematics_training_set,
            num_basis_vectors=num_basis_vectors,
            error=noise_level_flat,
            expander=NullExpander(),
            mean_translation=True
        )]
        basis_sum = BasisSum(names,bases)
        dimension = [{'systematics' : np.arange(1,num_sys_vectors)}]
        priors = {"systematics_prior" : bases[0].gaussian_prior} 
        meta_fitter = MetaFitter(basis_sum, temperatures_flat, noise_level_flat, quantity, quantity.name, *dimension,**priors,verbose=False)      # creates the meta_fitter object, which is an essential Pylinex object
        logZ_grid = meta_fitter["log_evidence"]

        best_indices = np.unravel_index(
            np.argmax(logZ_grid),
            logZ_grid.shape
        )

        fitter = meta_fitter.fitter_from_indices(best_indices)

        # print("Best term numbers:", fitter.sizes)
        # print("Maximum log evidence:", fitter.log_evidence)
                                                                    
        num_systematics_terms = fitter.sizes["systematics"]                                                
        # plot for the signal extraction
        pylinex_error = fitter.subbasis_channel_error("systematics")
        radiometer_error = noise[0]
        # plt.plot((fitter.subbasis_channel_mean("systematics")-temperatures_flat)/noise_level_flat,label = "residual fit mean",color="blue")
        # print(f"systematic terms used: {num_systematics_terms}")
        # print(f"reduced chi squared: {fitter.reduced_chi_squared}")
        # print(f"reduced psi squared: {fitter.psi_squared}")
        chi_squared = fitter.reduced_chi_squared
        psi_squared = fitter.psi_squared
        term_number = num_systematics_terms

        master_chi_squared_array[amplitude_index][index] = chi_squared
        master_psi_squared_array[amplitude_index][index] = psi_squared
        master_term_number_array[amplitude_index][index]= term_number   
        master_evidence_array[amplitude_index][index]= fitter.log_evidence

            ##### Lambda CDM extractions
        data=static_monchrome_beam_simulation[0]
        noise = static_monchrome_beam_simulation[5]
        signal = static_monchrome_beam_simulation[1]
        systematics = static_monchrome_beam_simulation[2]
        test_extraction_static_monochrome_LCDM = pylinex_extraction(systematics_training_set,signal_training_set,data,noise,signal,frequencies,IC="log_evidence",verbose=verbose,plot=plot,plot_residual=False,num_basis_vectors=num_basis_vectors,num_sys_vectors = num_sys_vectors, \
                        num_sig_vectors = num_sig_vectors, title = "Pylinex Extraction", ignore_IC = False,ylim=None,man_sys_terms=manual_systematics_terms,man_sig_terms=manual_signal_terms,man_sys_terms_lower=0,man_sig_terms_lower=0,multi_spectra=True,priors=True,covariance_expansion_factor=1\
                            ,evidence_cutoff=evidence_cutoff)



        master_chi_squared_array_CDM[amplitude_index][index] = test_extraction_static_monochrome_LCDM[0].reduced_chi_squared
        master_psi_squared_array_CDM[amplitude_index][index]  = test_extraction_static_monochrome_LCDM[0].psi_squared
        master_term_number_array_CDM[amplitude_index][index] = test_extraction_static_monochrome_LCDM[0].sizes["systematics"]
        master_term_number_array_CDM_signal[amplitude_index][index]  = test_extraction_static_monochrome_LCDM[0].sizes["signal"]
        master_evidence_array_CDM[amplitude_index][index]= test_extraction_static_monochrome_LCDM[0].log_evidence


    plt.xlabel(r"$\Delta$ ln Z")
    plt.ylabel("counts")
    plt.title(f"Noise level of {N_antenna}: Standard Deviation of {(master_evidence_array-master_evidence_array_CDM).std():.2f} and mean of {(master_evidence_array-master_evidence_array_CDM).mean():.2f}")
    f = h5py.File(f"/home/dbarker7752/21_cm_group/Exotic_Models_Data/expected_distro_vs_Lambda_extract_{N_antenna}ants_{constraint}_Bayesian_{N_extractions}_amplitude_gridsearch_top","w")
    f.create_dataset("chi squared",data=master_chi_squared_array)
    f.create_dataset("psi squared",data=master_psi_squared_array)
    f.create_dataset("term number",data=master_term_number_array)
    f.create_dataset("Bayesian evidence",data=master_evidence_array)
    f.create_dataset("chi squared CDM",data=master_chi_squared_array_CDM)
    f.create_dataset("psi squared CDM",data=master_psi_squared_array_CDM)
    f.create_dataset("foreground term number CDM",data=master_term_number_array_CDM)
    f.create_dataset("signal term number CDM",data=master_term_number_array_CDM_signal)
    f.create_dataset("Bayesian evidence CDM",data=master_evidence_array_CDM)
    f.close()

for i,n in zip(amplitudes,range(len(master_evidence_array))):
    log_diff = master_evidence_array[n]-master_evidence_array_CDM[n]
    percent_detection = np.where(log_diff < -5)[0].shape[0]/N_extractions
    percent_detection_null = np.where(log_diff > 5)[0].shape[0]/N_extractions
    plt.scatter(i,percent_detection_null*100,color=OFG_color)
    print(f"CDM detection = {percent_detection*100}")
    print(f"Null detection = {percent_detection_null*100}")

plt.xlabel("amplitude multiplier")
plt.ylabel("percent detection")
plt.title(f"{constraint_label} Foreground Amplitude Grid Search")


#### Comparing DMAN and CDM

In [ ]:

####  ####

### User Variables ###

constraint_value = 0.00025
constraint_label = "0.1%"
# constraints = np.array([0.00000025])
N_antenna = 10
dnu=1e6
dt=100*3600
N_extractions = 100
N_TS = 10000   # Number of curves in the training set
amplitudes = np.arange(0,5,0.1)

evidence_cutoff = 0

### Code Backend ###
plot=False
verbose = False
n_LSTs = 10 # this should match the number of foreground_training_sets you've made
## Spectral Index Map Variable
n_regions = 8   # number of regions that are created for your multi-region model
temp1 = 20     # lower frequency that is used for the temperature for the spectral index of the spectral index map
temp2 = 50     # upper frequency
f_low = temp1   # Only concerns the residual maps that are displayed through spectral_index_map_visualizations (not important for this)
f_high = temp2  # Only concerns the residual maps that are displayed through spectral_index_map_visualizations (not important for this)
NSIDE = 32
## Monochromatic Beam Variables.
N_beam = 1   # Number of beams to create. I'm not varying the beam right now, but this may be helpful later.
low_std = 0.5
high_std = low_std
## Synchrotron Foreground Variables
ylim = (-0.07,0.03)

quantity = AttributeQuantity("log_evidence")



constraint=str(constraint_value)[2:]

parameter_variation = [0,constraint_value,0]
sky_maps = master_foreground_array
sky_maps_truncated = sky_maps[:,4:49,:]  # the trucated sky maps that map the frequency range of frequencies (though it may not match the number of bins)
frequency_range_truncated = np.arange(5,50)  # Truncated frequency range that matches the sky map frequencies
# Pylinex Extraction Variables
manual_systematics_terms = 0
manual_signal_terms = 0
num_sig_vectors = 3 # maximum number of signal modes to use in the extractions
num_sys_vectors = 75    # maximum number of systematic modes to use in the extraction
num_basis_vectors = 75  # number of basis vectors to create for the root bases (make sure its as big as the largest num_sys or num_sig_vectors number)
## signal that is used as the input to the observation data: ##
parameter_array = np.array([[omR0,0],[omM0h2,0],[omK0,0],[omL0,0],[omB0h2,0/h**2],[H0,0]])

##  ##
signal_training_set = lambdaCDM_training_set[:,16::4]  ### indexing is needed to get it into the correct shape for the extraction. Training set used in the pylinex fit
IC = "DIC"

master_chi_squared_array = np.zeros((len(amplitudes),N_extractions))
master_psi_squared_array = np.zeros((len(amplitudes),N_extractions))
master_term_number_array =np.zeros((len(amplitudes),N_extractions))
master_evidence_array = np.zeros((len(amplitudes),N_extractions))
master_term_number_array_signal =np.zeros((len(amplitudes),N_extractions))

# array for the Lambda CDM extraction (not the null signal, that's the ones above) 
master_chi_squared_array_CDM = np.zeros((len(amplitudes),N_extractions))
master_psi_squared_array_CDM = np.zeros((len(amplitudes),N_extractions))
master_term_number_array_CDM = np.zeros((len(amplitudes),N_extractions))
master_evidence_array_CDM = np.zeros((len(amplitudes),N_extractions))
master_term_number_array_CDM_signal = np.zeros((len(amplitudes),N_extractions))


#Step 1: Create the spectral index map that will govern the B_values of the beam (since they govern the indices of the regions).
# NPIX = hp.nside2npix(NSIDE)
# spectral_maps = np.zeros((n_LSTs,NPIX))
# for l in range(n_LSTs):
#     spectral_map = py21cmsig.spectral_index_map(f_low,f_high,n_regions,temp1,temp2,NSIDE,ULSA_direction_32 = master_foreground_array[l])
#     spectral_maps[l] = spectral_map

# f = h5py.File("/home/dbarker7752/21_cm_group/Exotic_Models_Data/spectral_index_map_10to50MHz_32bit","w")
# f.create_dataset("spectral maps",data=spectral_maps)
# f.close()

if NSIDE != 32:
    raise ValueError ("The saved spectral maps are only 32 bit right now.")
hdf5 = h5py.File("/home/dbarker7752/21_cm_group/Exotic_Models_Data/spectral_index_map_10to50MHz_32bit","r")
spectral_maps = np.array(hdf5["spectral maps"])   # I'v visually varified that these make sense

BTS_curves_monochromaticbeam = np.zeros((n_LSTs,len(frequencies)))
# # Step 2: Create the monochromatic beam
for l in np.arange(n_LSTs):
    BTS_curves_monochromaticbeam_raw = (py21cmsig.gaussian_beams(frequency_range_truncated,low_std,resolution=NSIDE,monochromatic_mode=True)[0]*sky_maps_truncated).sum(axis=2)
    BTS_interpolator = scipy.interpolate.CubicSpline(frequency_range_truncated,BTS_curves_monochromaticbeam_raw[l])
    BTS_curves_monochromaticbeam[l] = BTS_interpolator(frequencies)

# creates the B values
N=1
monochromatic_gaussian_B_values = np.zeros((n_LSTs,N,len(frequencies),n_regions))
gaussian_beam_element = py21cmsig.gaussian_beams(frequencies,0.5,NSIDE,monochromatic_mode=True)

for l in range(n_LSTs):
    patch=perses.models.PatchyForegroundModel(frequencies,spectral_maps[l],n_regions) # define the regional patches
    region_indices = patch.foreground_pixel_indices_by_region_dictionary
    for r in range(n_regions):
        monochromatic_gaussian_B_values[l,0,:,r] = gaussian_beam_element[:,region_indices[r]].sum(axis=1)
# monochromatic_gaussian_B_values = py21cmsig.create_monochromatic_gaussian_beam_training_set(frequencies,N_beam,[low_std,high_std],n_LSTs,n_regions,NSIDE,spectral_maps,plot_extremes = True)[0]
monochromatic_gaussian_parameters = np.array([low_std])
# if len(frequencies) != monochromatic_gaussian_B_values[0][0].shape[0]:
#     raise ValueError ("Your loaded B_values do not have the same length as your frequencies. This will cause problems later")

# # Step 3: Create the beam-weighted foreground training set:
# # Step 3a: Create the BTS_curves to input to the foreground function:

# simulated_foreground_array sets the "true sky" observations that will be compared to the training sets for the extraction. It's your data, basically. 
parameter_variation_true = [0,0,0]   
simulated_foreground_array_10LSTs = np.zeros((n_LSTs,len(frequencies)))
for l in range(n_LSTs):
    simulated_foreground_array_10LSTs[l,:] = py21cmsig.synchrotron_foreground_updated_given_Bvalues(1,n_regions,frequencies,spectral_maps[l],sky_maps[l],parameter_variation_true,\
                                                                                    np.array([monochromatic_gaussian_B_values[l,0]]),np.array([low_std]),\
                                                                                    np.array([BTS_curves_monochromaticbeam[l]]))[0]

# The systematics training set itself
monochro_beam_TS_10LSTs = np.zeros((N_TS,n_LSTs,len(frequencies)))
for l in range(n_LSTs):
    monochro_beam_TS_10LSTs[:,l,:] = py21cmsig.synchrotron_foreground_updated_given_Bvalues(N_TS,n_regions,frequencies,spectral_maps[l],sky_maps[l],parameter_variation,\
                                                                                    np.array([monochromatic_gaussian_B_values[l,0]]),np.array([monochromatic_gaussian_parameters[0]]),\
                                                                                    np.array([BTS_curves_monochromaticbeam[l]]))[0]



    #Pylinex Extractions



    extraction_dict = {}

    ### Pylinex part (doing it manually since we are only using the foreground for this) ###
# for amplitude_index in tqdm(np.arange(len(amplitudes))):
for amplitude_index in np.arange(len(amplitudes)):
    amplitude = amplitudes[amplitude_index]
    print(f"Beginning processing of {amplitude}x amplitude multiplier.")
    for index in tqdm(np.arange(N_extractions)):  
    
        input_signal = -fiducial_signal*amplitude
        foreground = monochro_beam_TS_10LSTs[np.random.randint(N_TS)]
        static_monchrome_beam_simulation=py21cmsig.multi_spectra_simulation_run(frequencies,foreground,input_signal,N_antenna,dnu,dt)
        # data=static_monchrome_beam_simulation[0]
        # noise = static_monchrome_beam_simulation[5]
        # signal = static_monchrome_beam_simulation[1]
        # systematics = static_monchrome_beam_simulation[2]
        # systematics_training_set = monochro_beam_TS_10LSTs
        # flattened_systematics_training_set = np.reshape(systematics_training_set, (len(systematics_training_set), -1))
        # temperatures_flat = data.flatten()
        # noise_level_flat = noise.flatten()
        # Ns = len(noise)      # grabs the number of correlated spectra you are using
        # names = ["systematics"]        # names of your bases
        # bases = [
        # TrainedBasis(
        #     training_set=flattened_systematics_training_set,
        #     num_basis_vectors=num_basis_vectors,
        #     error=noise_level_flat,
        #     expander=NullExpander(),
        #     mean_translation=True
        # )]
        # basis_sum = BasisSum(names,bases)
        # dimension = [{'systematics' : np.arange(1,num_sys_vectors)}]
        # priors = {"systematics_prior" : bases[0].gaussian_prior} 
        # meta_fitter = MetaFitter(basis_sum, temperatures_flat, noise_level_flat, quantity, quantity.name, *dimension,**priors,verbose=False)      # creates the meta_fitter object, which is an essential Pylinex object
        # logZ_grid = meta_fitter["log_evidence"]

        # best_indices = np.unravel_index(
        #     np.argmax(logZ_grid),
        #     logZ_grid.shape
        # )

        # fitter = meta_fitter.fitter_from_indices(best_indices)

        # # print("Best term numbers:", fitter.sizes)
        # # print("Maximum log evidence:", fitter.log_evidence)
                                                                    
        # num_systematics_terms = fitter.sizes["systematics"]                                                
        # # plot for the signal extraction
        # pylinex_error = fitter.subbasis_channel_error("systematics")
        # radiometer_error = noise[0]
        # # plt.plot((fitter.subbasis_channel_mean("systematics")-temperatures_flat)/noise_level_flat,label = "residual fit mean",color="blue")
        # # print(f"systematic terms used: {num_systematics_terms}")
        # # print(f"reduced chi squared: {fitter.reduced_chi_squared}")
        # # print(f"reduced psi squared: {fitter.psi_squared}")
        # chi_squared = fitter.reduced_chi_squared
        # psi_squared = fitter.psi_squared
        # term_number = num_systematics_terms

        # master_chi_squared_array[amplitude_index][index] = chi_squared
        # master_psi_squared_array[amplitude_index][index] = psi_squared
        # master_term_number_array[amplitude_index][index]= term_number   
        # master_evidence_array[amplitude_index][index]= fitter.log_evidence

        signal_training_set_exotic = DMAN_training_set_curves[9000::]

        data=static_monchrome_beam_simulation[0]
        noise = static_monchrome_beam_simulation[5]
        signal = static_monchrome_beam_simulation[1]
        systematics = static_monchrome_beam_simulation[2]
        test_extraction_static_monochrome_LCDM = pylinex_extraction(systematics_training_set,signal_training_set_exotic,data,noise,signal,frequencies,IC="log_evidence",verbose=verbose,plot=plot,plot_residual=False,num_basis_vectors=num_basis_vectors,num_sys_vectors = num_sys_vectors, \
                        num_sig_vectors = num_sig_vectors, title = "Pylinex Extraction", ignore_IC = False,ylim=None,man_sys_terms=manual_systematics_terms,man_sig_terms=manual_signal_terms,man_sys_terms_lower=0,man_sig_terms_lower=0,multi_spectra=True,priors=True,covariance_expansion_factor=1\
                            ,evidence_cutoff=evidence_cutoff)



        master_chi_squared_array[amplitude_index][index] = test_extraction_static_monochrome_LCDM[0].reduced_chi_squared
        master_psi_squared_array[amplitude_index][index]  = test_extraction_static_monochrome_LCDM[0].psi_squared
        master_term_number_array[amplitude_index][index] = test_extraction_static_monochrome_LCDM[0].sizes["systematics"]
        master_term_number_array_signal[amplitude_index][index]  = test_extraction_static_monochrome_LCDM[0].sizes["signal"]
        master_evidence_array[amplitude_index][index]= test_extraction_static_monochrome_LCDM[0].log_evidence

            ##### Lambda CDM extractions
        data=static_monchrome_beam_simulation[0]
        noise = static_monchrome_beam_simulation[5]
        signal = static_monchrome_beam_simulation[1]
        systematics = static_monchrome_beam_simulation[2]
        test_extraction_static_monochrome_LCDM = pylinex_extraction(systematics_training_set,signal_training_set,data,noise,signal,frequencies,IC="log_evidence",verbose=verbose,plot=plot,plot_residual=False,num_basis_vectors=num_basis_vectors,num_sys_vectors = num_sys_vectors, \
                        num_sig_vectors = num_sig_vectors, title = "Pylinex Extraction", ignore_IC = False,ylim=None,man_sys_terms=manual_systematics_terms,man_sig_terms=manual_signal_terms,man_sys_terms_lower=0,man_sig_terms_lower=0,multi_spectra=True,priors=True,covariance_expansion_factor=1\
                            ,evidence_cutoff=evidence_cutoff)



        master_chi_squared_array_CDM[amplitude_index][index] = test_extraction_static_monochrome_LCDM[0].reduced_chi_squared
        master_psi_squared_array_CDM[amplitude_index][index]  = test_extraction_static_monochrome_LCDM[0].psi_squared
        master_term_number_array_CDM[amplitude_index][index] = test_extraction_static_monochrome_LCDM[0].sizes["systematics"]
        master_term_number_array_CDM_signal[amplitude_index][index]  = test_extraction_static_monochrome_LCDM[0].sizes["signal"]
        master_evidence_array_CDM[amplitude_index][index]= test_extraction_static_monochrome_LCDM[0].log_evidence


    plt.xlabel(r"$\Delta$ ln Z")
    plt.ylabel("counts")
    plt.title(f"Noise level of {N_antenna}: Standard Deviation of {(master_evidence_array-master_evidence_array_CDM).std():.2f} and mean of {(master_evidence_array-master_evidence_array_CDM).mean():.2f}")
    f = h5py.File(f"/home/dbarker7752/21_cm_group/Exotic_Models_Data/expected_distro_vs_Lambda_extract_{N_antenna}ants_{constraint}_Bayesian_{N_extractions}_amplitude_gridsearch_DMAN_top","w")
    f.create_dataset("chi squared",data=master_chi_squared_array)
    f.create_dataset("psi squared",data=master_psi_squared_array)
    f.create_dataset("term number",data=master_term_number_array)
    f.create_dataset("Bayesian evidence",data=master_evidence_array)
    f.create_dataset("chi squared CDM",data=master_chi_squared_array_CDM)
    f.create_dataset("psi squared CDM",data=master_psi_squared_array_CDM)
    f.create_dataset("foreground term number CDM",data=master_term_number_array_CDM)
    f.create_dataset("signal term number CDM",data=master_term_number_array_CDM_signal)
    f.create_dataset("Bayesian evidence CDM",data=master_evidence_array_CDM)
    f.close()

for i,n in zip(amplitudes,range(len(master_evidence_array))):
    log_diff = master_evidence_array[n]-master_evidence_array_CDM[n]
    percent_detection = np.where(log_diff < -5)[0].shape[0]/N_extractions
    percent_detection_null = np.where(log_diff > 5)[0].shape[0]/N_extractions
    plt.scatter(i,percent_detection_null*100,color=OFG_color)
    print(f"CDM detection = {percent_detection*100}")
    print(f"Null detection = {percent_detection_null*100}")

plt.xlabel("amplitude multiplier")
plt.ylabel("percent detection")
plt.title(f"{constraint_label} Foreground Amplitude Grid Search")


In [ ]:
for i,n in zip(amplitudes,range(len(master_evidence_array))):
    log_diff = master_evidence_array[n]-master_evidence_array_CDM[n]
    percent_detection = np.where(log_diff < -5)[0].shape[0]/N_extractions
    percent_detection_null = np.where(log_diff > 5)[0].shape[0]/N_extractions
    plt.scatter(i,percent_detection*100,color=OFG_color)
    print(f"CDM detection = {percent_detection*100}")
    print(f"Null detection = {percent_detection_null*100}")